# Qwen3.5-2B Fine-tuning with Unsloth — Syrian Medical Records

- `load_in_4bit = True` to stay within 4 GB VRAM
- LoRA `r=8` → ~0.52% trainable parameters
- Tokenizer loaded via `AutoTokenizer` to avoid vision processor
- Saves the full merged model to `./qwen_syrian_finetuned/`

In [ ]:
import os, importlib.util

!pip uninstall -qqq -y torchcodec sentence-transformers 2>/dev/null || true

!pip install -qqq bitsandbytes

!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy; _numpy = f"numpy=={numpy.__version__}"
    except: _numpy = "numpy"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

!uv pip install --upgrade --no-deps trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0

!python -m bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
import torch

model, _ = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3.5-2B",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3.5-2B")
tokenizer.pad_token = tokenizer.eos_token


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    target_modules = "all-linear",
    use_rslora = False,
    loftq_config = None,
    random_state = 3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} = {trainable/total*100:.3f}%")


Trainable: 11,555,328 / 1,387,443,520 = 0.833%


In [ ]:
import json
from datasets import Dataset

with open("/content/syrianRecords.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Loaded {len(raw)} records")

SYSTEM = (
    "أنت مساعد طبي متخصص في استخراج البيانات من النصوص الصوتية الطبية السورية. "
    "استخرج الحقول المطلوبة وأعدها بصيغة JSON فقط."
)

def format_record(record):
    user_msg      = record["voice_text"]
    assistant_msg = json.dumps(record["extracted_fields"], ensure_ascii=False)
    return (
        f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
    )

formatted_texts = [format_record(r) for r in raw]

tokenized = tokenizer(
    formatted_texts,
    truncation = True,
    max_length = 512,
    padding = False,
)

dataset = Dataset.from_dict({
    "input_ids":      tokenized["input_ids"],
    "attention_mask": tokenized["attention_mask"],
    "labels":         tokenized["input_ids"],
})

print(f"Dataset size: {len(dataset)}")
print(f"Example token count: {len(dataset[0]['input_ids'])}")


Loaded 1599 records
Dataset size: 1599
Example token count: 136


In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,

        num_train_epochs = 1,
        warmup_ratio = 0.05,

        learning_rate      = 2e-4,
        optim              = "adamw_8bit",
        lr_scheduler_type  = "cosine",
        weight_decay       = 0.01,

        logging_steps  = 10,
        seed           = 3407,
        output_dir     = "outputs",
        report_to      = "none",

        max_seq_length          = 512,
        remove_unused_columns   = False,
        dataset_text_field      = "",
        dataset_kwargs          = {"skip_prepare_dataset": True},
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"GPU: {gpu_stats.name}  |  Max VRAM: {round(gpu_stats.total_memory/1024**3,2)} GB")
print(f"Reserved before training: {start_mem} GB")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Switching to float32 training since model cannot work with float16
GPU: Tesla T4  |  Max VRAM: 14.56 GB
Reserved before training: 3.68 GB


In [ ]:
trainer_stats = trainer.train()

used    = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_mem = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3)
print(f"\nTraining time : {trainer_stats.metrics['train_runtime']:.1f}s "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Peak VRAM     : {used} GB / {max_mem} GB  ({used/max_mem*100:.1f}%)")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,599 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 11,555,328 of 2,224,796,992 (0.52% trained)


Step,Training Loss
10,2.695565
20,1.293030
30,0.613588
40,0.418553
50,0.315358
60,0.258337
70,0.223438
80,0.201116
90,0.181212
100,0.176551



Training time : 2372.4s (39.5 min)
Peak VRAM     : 3.68 GB / 14.563 GB  (25.3%)


In [ ]:
FastLanguageModel.for_inference(model)

test_text = raw[0]["voice_text"]
print("Input:", test_text)
print("Expected:", json.dumps(raw[0]["extracted_fields"], ensure_ascii=False))
print("\nModel output:")

prompt = (
    f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
    f"<|im_start|>user\n{test_text}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    input_ids,
    streamer       = streamer,
    max_new_tokens = 200,
    use_cache      = True,
    temperature    = 0.1,
    do_sample      = True,
)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Input: العين التنتين سوا مشتبه بإصابته. هاد المريض موظف حكومي من النبك وعمره خمسين.
Expected: {"جهة العين": "كلتا العينين", "الجنس": "أنثى", "العمر": 50, "المدينة": "النبك", "الحالة": "مشتبه به", "المهنة": "موظف حكومي", "ملاحظات": null}

Model output:
{"جهة العين": "كلتا العينين", "الجنس": "أنثى", "العمر": 50, "المدينة": "النبك", "الحالة": "مشتبه به", "المهنة": "موظف حكومي", "ملاحظات": null}<|im_end|>
